In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta

11_02 --> 17_02

Read the csv files

In [2]:
data_courant = pd.read_csv("De 11_02 à 17_02 2022\Courant\COURANT QA.csv", encoding='utf-16',sep="\t") 
data_Debit_eau = pd.read_csv("De 11_02 à 17_02 2022\Débit Eau\DEBIT EAU QA.csv",encoding='utf-16',sep="\t") 
data_Debit_solide = pd.read_csv("De 11_02 à 17_02 2022\Débit Solide\DEBIT SOLIDE QA.csv",encoding='utf-16', sep= '\t') 
data_refus_tamis = pd.read_excel("De 11_02 à 17_02 2022\Refus Tamis\TB_D80_et_performances_Usine_Février_2022.xlsx")
data_puissance = pd.read_csv("De 11_02 à 17_02 2022\Puissance\PUISSANCE QA.csv",encoding='utf-16',sep="\t") 

In [3]:
print("-----------------Courant-------------------------------")
print(data_courant)
print("-----------------Debit eau-------------------------------")
print(data_Debit_eau)
print("-----------------Debit solide-------------------------------")
print(data_Debit_solide)
print("-----------------Refus tamis-------------------------------")
print(data_refus_tamis.head(10))
print("-----------------Puissance-------------------------------")
print(data_puissance)

-----------------Courant-------------------------------
               COURANT Time    COURANT ValueY
0       11/02/2022 00:00:00  69,7350311279297
1       11/02/2022 00:00:00  68,7320556640625
2       11/02/2022 00:00:01  64,6725997924805
3       11/02/2022 00:00:01  60,7117309570313
4       11/02/2022 00:00:02  61,8302955627441
...                     ...               ...
831950  17/02/2022 15:00:53  63,2038459777832
831951  17/02/2022 15:00:54  68,2458801269531
831952  17/02/2022 15:00:54  66,4371337890625
831953  17/02/2022 15:00:55  68,4056701660156
831954  17/02/2022 15:00:55  71,5267639160156

[831955 rows x 2 columns]
-----------------Debit eau-------------------------------
             DEBIT EAU Time  DEBIT EAU ValueY
0       11/02/2022 00:00:01  9,69690418243408
1       11/02/2022 00:00:03  9,80179405212402
2       11/02/2022 00:00:05  9,86689853668213
3       11/02/2022 00:00:07  9,80541133880615
4       11/02/2022 00:00:09  9,86689853668213
...                     ...    

Function to pre-process "refus tamis"

In [2]:
def process_refu_tamis(data):
    data.dropna(how='all',inplace=True)
    del data["Unnamed: 0"]
    data.rename(columns={'Unnamed: 1':"Jours", 'Unnamed: 2':"Poste",'Unnamed: 3':"Refus Tamis +500 en % QA"}, inplace=True)
    data.drop(index=1,inplace=True)
    for i,line in enumerate(data.index):
        if data.iloc[i]["Jours"] is np.nan:
            data.iloc[i]["Jours"] = data.iloc[i-1]["Jours"]

In [5]:
process_refu_tamis(data_refus_tamis)
data_refus_tamis

,Jours,Poste,Refus Tamis +500 en % QA
3,2022-02-01 00:00:00,1,26.205511
4,2022-02-01 00:00:00,2,27.455296
5,2022-02-01 00:00:00,3,29.126214
6,2022-02-02 00:00:00,1,18.786127
7,2022-02-02 00:00:00,2,23.913043
...,...,...,...
85,2022-02-28 00:00:00,2,33.198507
86,2022-02-28 00:00:00,3,28.062157
87,MOYENNE,NaN,24.345225
88,MOYENNE,NaN,15.627498


Print the types of every column in our data

In [6]:
print(data_courant.dtypes)
print(data_Debit_eau.dtypes)
print(data_Debit_solide.dtypes)
print(data_refus_tamis.dtypes)
print(data_puissance.dtypes)

COURANT Time      object
COURANT ValueY    object
dtype: object
DEBIT EAU Time      object
DEBIT EAU ValueY    object
dtype: object
SOLIDE Time       object
SOLIDE ValueY     object
SP_INT Time       object
SP_INT ValueY    float64
dtype: object
Jours                       object
Poste                       object
Refus Tamis +500 en % QA    object
dtype: object
PUISSANCE Time      object
PUISSANCE ValueY    object
dtype: object


Convert the types from object to date_time

In [7]:
data_courant['COURANT Time'] = pd.to_datetime(data_courant['COURANT Time'], format="%d/%m/%Y %H:%M:%S")
data_Debit_eau['DEBIT EAU Time'] = pd.to_datetime(data_Debit_eau['DEBIT EAU Time'], format="%d/%m/%Y %H:%M:%S")
data_Debit_solide['SOLIDE Time'] = pd.to_datetime(data_Debit_solide['SOLIDE Time'], format="%d/%m/%Y %H:%M:%S")
data_puissance['PUISSANCE Time'] = pd.to_datetime(data_puissance['PUISSANCE Time'], format="%d/%m/%Y %H:%M:%S")
#data_refus_tamis['Jours'] = pd.to_datetime(data_refus_tamis['Jours'], format="%Y/%m/%d %H:%M:%S")

In [8]:
print(data_courant.dtypes)
print(data_Debit_eau.dtypes)
print(data_Debit_solide.dtypes)
print(data_puissance.dtypes)
#print(data_refus_tamis.dtypes)

COURANT Time      datetime64[ns]
COURANT ValueY            object
dtype: object
DEBIT EAU Time      datetime64[ns]
DEBIT EAU ValueY            object
dtype: object
SOLIDE Time      datetime64[ns]
SOLIDE ValueY            object
SP_INT Time              object
SP_INT ValueY           float64
dtype: object
PUISSANCE Time      datetime64[ns]
PUISSANCE ValueY            object
dtype: object


Replace the "," by "."

In [9]:
data_courant["COURANT ValueY"] = data_courant["COURANT ValueY"].apply(lambda x: x.replace("," , "."))
data_Debit_eau["DEBIT EAU ValueY"] = data_Debit_eau["DEBIT EAU ValueY"].apply(lambda x: x.replace("," , "."))
data_Debit_solide["SOLIDE ValueY"] = data_Debit_solide["SOLIDE ValueY"].apply(lambda x: x.replace("," , "."))
data_puissance["PUISSANCE ValueY"] = data_puissance["PUISSANCE ValueY"].apply(lambda x: x.replace("," , "."))

Change the type from "object" to "float" for the Value Columns

In [10]:
data_courant["COURANT ValueY"] = data_courant["COURANT ValueY"].astype(float)
data_Debit_eau["DEBIT EAU ValueY"] = data_Debit_eau["DEBIT EAU ValueY"].astype(float)
data_Debit_solide["SOLIDE ValueY"] = data_Debit_solide["SOLIDE ValueY"].astype(float)
data_puissance["PUISSANCE ValueY"] = data_puissance["PUISSANCE ValueY"].astype(float)

Shape of every csv file

In [11]:
print(data_courant.shape)
print(data_Debit_eau.shape)
print(data_Debit_solide.shape)
print(data_refus_tamis.shape)
print(data_puissance.shape)

(831955, 2)
(207520, 2)
(207884, 4)
(87, 3)
(831686, 2)


The max and the min

In [12]:
print("Courant : ")
print(data_courant['COURANT Time'].min())
print(data_courant['COURANT Time'].max())
print("Débit Eau")
print(data_Debit_eau['DEBIT EAU Time'].min())
print(data_Debit_eau['DEBIT EAU Time'].max())
print("Débit Solide")
print(data_Debit_solide['SOLIDE Time'].min())
print(data_Debit_solide['SOLIDE Time'].max())
print("Puissance")
print(data_puissance['PUISSANCE Time'].min())
print(data_puissance['PUISSANCE Time'].max())

Courant : 
2022-02-11 00:00:00
2022-02-17 15:00:55
Débit Eau
2022-02-11 00:00:01
2022-02-17 15:00:04
Débit Solide
2022-02-11 00:00:01
2022-02-17 15:00:50
Puissance
2022-02-11 00:00:00
2022-02-17 15:00:55


In [13]:
interv2 = data_courant[data_courant["COURANT Time"].between(pd.to_datetime("2022-02-17 11:10:00"),pd.to_datetime("2022-02-17 11:11:00"))]
print(interv2["COURANT ValueY"].mean())
print(interv2.shape[0])

65.16948268452629
122


In [6]:
def second_to_minute(data):
    columns = data.columns
    time_start = pd.to_datetime("2022-02-11 00:00:00")
    time_end = pd.to_datetime("2022-02-17 15:00:00")
    list_time = []
    list_value = []
    list_value_2 = []
    while(time_start <= time_end):
        interv = data[data[columns[0]].between(time_start,time_start + timedelta(minutes=1))]
        if interv.shape[0] == 0:
            list_value.append(float("nan"))
        else:
            list_value.append(interv[columns[1]].mean())
        list_time.append(time_start)
        if len(columns) == 4:
            list_value_2.append(interv[columns[3]].mean())
        time_start = time_start + timedelta(minutes=1)
    if len(columns) == 2:
        new_data = pd.DataFrame(data=zip(list_time,list_value),columns=[columns[0],columns[1]])
    elif len(columns) == 4:
        new_data = pd.DataFrame(data=zip(list_time,list_value,list_value_2),columns=[columns[0],columns[1],columns[3]])
    return new_data

In [17]:
new_data_courant = second_to_minute(data_courant)
new_data_debit_eau = second_to_minute(data_Debit_eau)
new_data_debit_solide = second_to_minute(data_Debit_solide)
new_data_puissance = second_to_minute(data_puissance)

In [18]:
print(new_data_courant)
print(new_data_debit_eau)
print(new_data_debit_solide)
print(new_data_puissance)

            COURANT Time  COURANT ValueY
0    2022-02-11 00:00:00       65.442617
1    2022-02-11 00:01:00       65.784166
2    2022-02-11 00:02:00       65.831569
3    2022-02-11 00:03:00       64.970144
4    2022-02-11 00:04:00       65.341595
...                  ...             ...
9536 2022-02-17 14:56:00       66.063068
9537 2022-02-17 14:57:00       65.189325
9538 2022-02-17 14:58:00       66.403390
9539 2022-02-17 14:59:00       66.516785
9540 2022-02-17 15:00:00       66.288485

[9541 rows x 2 columns]
          DEBIT EAU Time  DEBIT EAU ValueY
0    2022-02-11 00:00:00          9.366681
1    2022-02-11 00:01:00          8.775439
2    2022-02-11 00:02:00          9.686415
3    2022-02-11 00:03:00          9.813006
4    2022-02-11 00:04:00          9.897039
...                  ...               ...
9536 2022-02-17 14:56:00          9.216323
9537 2022-02-17 14:57:00          9.203489
9538 2022-02-17 14:58:00          9.226007
9539 2022-02-17 14:59:00          9.297878
9540 2022-

Rename the columns of Time

In [32]:
new_data_courant.rename(columns={"COURANT Time": "Time"}, inplace=True)
new_data_debit_eau.rename(columns={"DEBIT EAU Time": "Time"}, inplace=True)
new_data_debit_solide.rename(columns={"SOLIDE Time": "Time"}, inplace=True)
new_data_puissance.rename(columns={"PUISSANCE Time": "Time"}, inplace=True)

Merge all this data(column) in on pandas dataframe 

In [33]:
data_11_02_a_17_02 = pd.merge(pd.merge(new_data_courant,new_data_debit_eau,on='Time'),pd.merge(new_data_debit_solide,new_data_puissance,on='Time'),on="Time")

In [34]:
data_11_02_a_17_02

,Time,COURANT ValueY,DEBIT EAU ValueY,SOLIDE ValueY,SP_INT ValueY,PUISSANCE ValueY
0,2022-02-11 00:00:00,65.442617,9.366681,115.298274,115.0,559.952023
1,2022-02-11 00:01:00,65.784166,8.775439,113.518156,115.0,562.290662
2,2022-02-11 00:02:00,65.831569,9.686415,113.878942,115.0,562.912594
3,2022-02-11 00:03:00,64.970144,9.813006,117.350561,115.0,557.878828
4,2022-02-11 00:04:00,65.341595,9.897039,113.183894,115.0,561.615206
...,...,...,...,...,...,...
9536,2022-02-17 14:56:00,66.063068,9.216323,111.649853,NaN,564.337326
9537,2022-02-17 14:57:00,65.189325,9.203489,110.982478,NaN,558.512230
9538,2022-02-17 14:58:00,66.403390,9.226007,109.076022,NaN,569.896756
9539,2022-02-17 14:59:00,66.516785,9.297878,109.423127,NaN,569.889818


In [36]:
data_11_02_a_17_02.to_csv("data_11_02_a_17_02.csv")

15_09 --> 20_10

Read csv files

In [37]:
data_courant_1_10 = pd.read_csv("De 15_09 à 20_10 2022\Courant\COURANT QA 1 - 10.csv", encoding='utf-16',sep="\t") 
data_courant_10_20_10 = pd.read_csv("De 15_09 à 20_10 2022\Courant\COURANT QA 10 - 20  10.csv", encoding='utf-16',sep="\t") 
data_courant_15_9_1_10 = pd.read_csv("De 15_09 à 20_10 2022\Courant\COURANT QA 15 9  - 1 10.csv", encoding='utf-16',sep="\t") 

data_Debit_eau_1_20_10 = pd.read_csv("De 15_09 à 20_10 2022\Débit Eau\DEBIT EAU QA 1 - 20 10.csv",encoding='utf-16',sep="\t") 
data_Debit_eau_15_9_1_10 = pd.read_csv("De 15_09 à 20_10 2022\Débit Eau\DEBIT EAU QA 15 9 - 1 10.csv",encoding='utf-16',sep="\t") 

data_Debit_solide_1_20_10 = pd.read_csv("De 15_09 à 20_10 2022\Débit Solide\DEBIT SOLIDE QA 1 - 20 10.csv",encoding='utf-16', sep= '\t') 
data_Debit_solide_15_9_1_10 = pd.read_csv("De 15_09 à 20_10 2022\Débit Solide\DEBIT SOLIDE QA 15 9 - 1 10.csv",encoding='utf-16', sep= '\t') 

data_refus_tamis_octobre = pd.read_excel("De 15_09 à 20_10 2022\Refus Tamis\TB_D80_et_performances_Usine_Octobre_2022.xlsx")
data_refus_tamis_septembre = pd.read_excel("De 15_09 à 20_10 2022\Refus Tamis\TB_D80_et_performances_Usine_Septembre_2022.xlsx")

data_puissance_QA_1 = pd.read_csv("De 15_09 à 20_10 2022\Puissance\P QA 1.csv",encoding='utf-16',sep="\t") 
data_puissance_QA_2 = pd.read_csv("De 15_09 à 20_10 2022\Puissance\P QA 2.csv",encoding='utf-16',sep="\t") 
data_puissance_QA_3 = pd.read_csv("De 15_09 à 20_10 2022\Puissance\P QA 3.csv",encoding='utf-16',sep="\t") 
data_puissance_QA_4 = pd.read_csv("De 15_09 à 20_10 2022\Puissance\P QA 4.csv",encoding='utf-16',sep="\t") 
data_puissance_QA_5 = pd.read_csv("De 15_09 à 20_10 2022\Puissance\P QA 5.csv",encoding='utf-16',sep="\t") 
data_puissance_QA_6 = pd.read_csv("De 15_09 à 20_10 2022\Puissance\P QA 6.csv",encoding='utf-16',sep="\t") 
data_puissance_QA_7 = pd.read_csv("De 15_09 à 20_10 2022\Puissance\P QA 7.csv",encoding='utf-16',sep="\t") 

In [49]:
print("-----------------Courant-------------------------------")
print(data_courant_1_10)
print(data_courant_10_20_10)
print(data_courant_15_9_1_10)

print("-----------------Debit eau-------------------------------")
print(data_Debit_eau_1_20_10)
print(data_Debit_eau_15_9_1_10)

print("-----------------Debit solide-------------------------------")
print(data_Debit_solide_1_20_10)
print(data_Debit_solide_15_9_1_10)

print("-----------------Refus tamis-------------------------------")
print(data_refus_tamis_octobre.head(10))
print(data_refus_tamis_septembre.head(10))

print("-----------------Puissance-------------------------------")
print(data_puissance_QA_1)
print(data_puissance_QA_2)
print(data_puissance_QA_3)
print(data_puissance_QA_4)
print(data_puissance_QA_5)
print(data_puissance_QA_6)
print(data_puissance_QA_7)

-----------------Courant-------------------------------
                COURANT Time    COURANT ValueY
0        01/10/2022 14:49:12  66,5901336669922
1        01/10/2022 14:49:12  63,8192291259766
2        01/10/2022 14:49:13  65,2097778320313
3        01/10/2022 14:49:13   59,011791229248
4        01/10/2022 14:49:14  67,0797119140625
...                      ...               ...
1559386  10/10/2022 15:49:09  63,9246215820313
1559387  10/10/2022 15:49:10  69,1196441650391
1559388  10/10/2022 15:49:10  62,0886840820313
1559389  10/10/2022 15:49:11  60,9837226867676
1559390  10/10/2022 15:49:11  60,3853454589844

[1559391 rows x 2 columns]
                COURANT Time    COURANT ValueY
0        10/10/2022 14:49:12  57,5158424377441
1        10/10/2022 14:49:12   60,963321685791
2        10/10/2022 14:49:13  55,4215126037598
3        10/10/2022 14:49:13  69,0856475830078
4        10/10/2022 14:49:14  55,5575103759766
...                      ...               ...
1734865  20/10/2022 15:

pre process refus tamis_octobre & septembre

In [40]:
process_refu_tamis(data_refus_tamis_octobre)
process_refu_tamis(data_refus_tamis_septembre)
print(data_refus_tamis_octobre)
print(data_refus_tamis_septembre)

                  Jours Poste Refus Tamis +500 en % QA
3   2022-10-01 00:00:00     1                22.182362
4   2022-10-01 00:00:00     2                19.904009
5   2022-10-01 00:00:00     3                      NaN
6   2022-10-02 00:00:00     1                      NaN
7   2022-10-02 00:00:00     2                      NaN
..                  ...   ...                      ...
94  2022-10-31 00:00:00     2                      NaN
95  2022-10-31 00:00:00     3                      NaN
96              MOYENNE   NaN                 20.71681
97              MOYENNE   NaN                12.694195
98              MOYENNE   NaN                34.245906

[96 rows x 3 columns]
                  Jours Poste Refus Tamis +500 en % QA
3   2022-09-01 00:00:00     1                    29.03
4   2022-09-01 00:00:00     2                    22.29
5   2022-09-01 00:00:00     3                      NaN
6   2022-09-02 00:00:00     1                      NaN
7   2022-09-02 00:00:00     2             

print the types of every column

In [42]:
print(data_courant_10_20_10.dtypes)
print(data_courant_1_10.dtypes)
print(data_courant_15_9_1_10.dtypes)

print(data_Debit_eau_1_20_10.dtypes)
print(data_Debit_eau_15_9_1_10.dtypes)

print(data_Debit_solide_1_20_10.dtypes)
print(data_Debit_solide_15_9_1_10.dtypes)

print(data_refus_tamis_octobre.dtypes)
print(data_refus_tamis_septembre.dtypes)

print(data_puissance_QA_1.dtypes)
print(data_puissance_QA_2.dtypes)
print(data_puissance_QA_3.dtypes)
print(data_puissance_QA_4.dtypes)
print(data_puissance_QA_5.dtypes)
print(data_puissance_QA_6.dtypes)
print(data_puissance_QA_7.dtypes)

COURANT Time      object
COURANT ValueY    object
dtype: object
COURANT Time      object
COURANT ValueY    object
dtype: object
COURANT Time      object
COURANT ValueY    object
dtype: object
DEBIT EAU Time      object
DEBIT EAU ValueY    object
dtype: object
DEBIT EAU Time      object
DEBIT EAU ValueY    object
dtype: object
SOLIDE Time      object
SOLIDE ValueY    object
dtype: object
SOLIDE Time      object
SOLIDE ValueY    object
dtype: object
Jours                       object
Poste                       object
Refus Tamis +500 en % QA    object
dtype: object
Jours                       object
Poste                       object
Refus Tamis +500 en % QA    object
dtype: object
PUISSANCE Time      object
PUISSANCE ValueY    object
dtype: object
PUISSANCE Time      object
PUISSANCE ValueY    object
dtype: object
PUISSANCE Time      object
PUISSANCE ValueY    object
dtype: object
PUISSANCE Time      object
PUISSANCE ValueY    object
dtype: object
PUISSANCE Time      object
PUISSANCE V

In [61]:
frames = (data_courant_15_9_1_10 , data_courant_1_10 , data_courant_10_20_10)
data_courant_15_9_a_20_10 = pd.concat(frames)
data_courant_15_9_a_20_10

,COURANT Time,COURANT ValueY
0,15/09/2022 14:49:12,"59,0287933349609"
1,15/09/2022 14:49:12,"60,0113563537598"
2,15/09/2022 14:49:13,"71,142578125"
3,15/09/2022 14:49:13,"70,0580139160156"
4,15/09/2022 14:49:14,"64,9411926269531"
...,...,...
1734865,20/10/2022 15:49:09,"57,8694305419922"
1734866,20/10/2022 15:49:10,"52,9939994812012"
1734867,20/10/2022 15:49:10,"60,677734375"
1734868,20/10/2022 15:49:11,"52,7254028320313"


In [62]:
frames_2 = (data_Debit_eau_15_9_1_10 , data_Debit_eau_1_20_10)
data_debit_eau_15_9_a_20_10 = pd.concat(frames_2)
data_debit_eau_15_9_a_20_10

,DEBIT EAU Time,DEBIT EAU ValueY
0,15/09/2022 14:45:20,"9,71137142181396"
1,15/09/2022 14:45:22,"10,076678276062"
2,15/09/2022 14:45:24,"10,3804979324341"
3,15/09/2022 14:45:26,"10,3370952606201"
4,15/09/2022 14:45:28,"10,1273145675659"
...,...,...
821980,20/10/2022 15:45:09,"9,375"
821981,20/10/2022 15:45:11,"9,24840831756592"
821982,20/10/2022 15:45:13,"9,10011577606201"
821983,20/10/2022 15:45:15,"8,95182228088379"


In [63]:
frames_3 = (data_Debit_solide_15_9_1_10 , data_Debit_solide_1_20_10)
data_debit_solide_15_9_a_20_10 = pd.concat(frames_3)
data_debit_solide_15_9_a_20_10

,SOLIDE Time,SOLIDE ValueY
0,15/09/2022 14:40:58,"115,134910583496"
1,15/09/2022 14:41:00,"114,447700500488"
2,15/09/2022 14:41:02,"115,297668457031"
3,15/09/2022 14:41:04,"115,315757751465"
4,15/09/2022 14:41:06,"115,315757751465"
...,...,...
822152,20/10/2022 15:40:47,"108,118125915527"
822153,20/10/2022 15:40:49,"107,828773498535"
822154,20/10/2022 15:40:51,"108,651626586914"
822155,20/10/2022 15:40:53,"108,190467834473"


In [64]:
frames_4 = (data_refus_tamis_octobre , data_refus_tamis_septembre)
data_refus_tamis_15_9_a_20_10 = pd.concat(frames_4)
data_refus_tamis_15_9_a_20_10

,Jours,Poste,Refus Tamis +500 en % QA
3,2022-10-01 00:00:00,1,22.182362
4,2022-10-01 00:00:00,2,19.904009
5,2022-10-01 00:00:00,3,NaN
6,2022-10-02 00:00:00,1,NaN
7,2022-10-02 00:00:00,2,NaN
...,...,...,...
91,2022-09-30 00:00:00,2,NaN
92,2022-09-30 00:00:00,3,NaN
93,MOYENNE,NaN,19.457157
94,MOYENNE,NaN,13.326848


In [65]:
frames_5 = (data_puissance_QA_1,data_puissance_QA_2,data_puissance_QA_3,data_puissance_QA_4,data_puissance_QA_5,data_puissance_QA_6,data_puissance_QA_7)
data_puissance_15_9_a_20_10 = pd.concat(frames_5)
data_puissance_15_9_a_20_10

,PUISSANCE Time,PUISSANCE ValueY
0,15/09/2022 14:49:12,"506,09228515625"
1,15/09/2022 14:49:12,"514,516418457031"
2,15/09/2022 14:49:13,"609,951599121094"
3,15/09/2022 14:49:13,"600,652954101563"
4,15/09/2022 14:49:14,"586,2822265625"
...,...,...
871195,20/10/2022 17:01:53,"557,774169921875"
871196,20/10/2022 17:01:54,"548,388061523438"
871197,20/10/2022 17:01:54,"456,509124755859"
871198,20/10/2022 17:01:55,"522,969787597656"


Convert the types from object to date_time

In [66]:
data_courant_15_9_a_20_10['COURANT Time'] = pd.to_datetime(data_courant_15_9_a_20_10['COURANT Time'], format="%d/%m/%Y %H:%M:%S")
data_debit_eau_15_9_a_20_10['DEBIT EAU Time'] = pd.to_datetime(data_debit_eau_15_9_a_20_10['DEBIT EAU Time'], format="%d/%m/%Y %H:%M:%S")
data_debit_solide_15_9_a_20_10['SOLIDE Time'] = pd.to_datetime(data_debit_solide_15_9_a_20_10['SOLIDE Time'], format="%d/%m/%Y %H:%M:%S")
data_puissance_15_9_a_20_10['PUISSANCE Time'] = pd.to_datetime(data_puissance_15_9_a_20_10['PUISSANCE Time'], format="%d/%m/%Y %H:%M:%S")
#data_refus_tamis_15_9_a_20_10['Jours'] = pd.to_datetime(data_refus_tamis_15_9_a_20_10['Jours'], format="%Y/%m/%d %H:%M:%S")

In [67]:
print(data_courant_15_9_a_20_10.dtypes)
print(data_debit_eau_15_9_a_20_10.dtypes)
print(data_debit_solide_15_9_a_20_10.dtypes)
print(data_puissance_15_9_a_20_10.dtypes)
#print(data_refus_tamis_15_9_a_20_10.dtypes)

COURANT Time      datetime64[ns]
COURANT ValueY            object
dtype: object
DEBIT EAU Time      datetime64[ns]
DEBIT EAU ValueY            object
dtype: object
SOLIDE Time      datetime64[ns]
SOLIDE ValueY            object
dtype: object
PUISSANCE Time      datetime64[ns]
PUISSANCE ValueY            object
dtype: object


Replace the "," by "."

In [68]:
data_courant_15_9_a_20_10["COURANT ValueY"] = data_courant_15_9_a_20_10["COURANT ValueY"].apply(lambda x: x.replace("," , "."))
data_debit_eau_15_9_a_20_10["DEBIT EAU ValueY"] = data_debit_eau_15_9_a_20_10["DEBIT EAU ValueY"].apply(lambda x: x.replace("," , "."))
data_debit_solide_15_9_a_20_10["SOLIDE ValueY"] = data_debit_solide_15_9_a_20_10["SOLIDE ValueY"].apply(lambda x: x.replace("," , "."))
data_puissance_15_9_a_20_10["PUISSANCE ValueY"] = data_puissance_15_9_a_20_10["PUISSANCE ValueY"].apply(lambda x: x.replace("," , "."))

Change the type from "object" to "float" for the Value Columns

In [69]:
data_courant_15_9_a_20_10["COURANT ValueY"] = data_courant_15_9_a_20_10["COURANT ValueY"].astype(float)
data_debit_eau_15_9_a_20_10["DEBIT EAU ValueY"] = data_debit_eau_15_9_a_20_10["DEBIT EAU ValueY"].astype(float)
data_debit_solide_15_9_a_20_10["SOLIDE ValueY"] = data_debit_solide_15_9_a_20_10["SOLIDE ValueY"].astype(float)
data_puissance_15_9_a_20_10["PUISSANCE ValueY"] = data_puissance_15_9_a_20_10["PUISSANCE ValueY"].astype(float)

Shape of every csv file

In [70]:
print(data_courant_15_9_a_20_10.shape)
print(data_debit_eau_15_9_a_20_10.shape)
print(data_debit_solide_15_9_a_20_10.shape)
print(data_refus_tamis_15_9_a_20_10.shape)
print(data_puissance_15_9_a_20_10.shape)

(6066261, 2)
(1514985, 2)
(1515157, 2)
(189, 3)
(6095061, 2)


The max and the min

In [71]:
print("Courant : ")
print(data_courant_15_9_a_20_10['COURANT Time'].min())
print(data_courant_15_9_a_20_10['COURANT Time'].max())
print("Débit Eau")
print(data_debit_eau_15_9_a_20_10['DEBIT EAU Time'].min())
print(data_debit_eau_15_9_a_20_10['DEBIT EAU Time'].max())
print("Débit Solide")
print(data_debit_solide_15_9_a_20_10['SOLIDE Time'].min())
print(data_debit_solide_15_9_a_20_10['SOLIDE Time'].max())
print("Puissance")
print(data_puissance_15_9_a_20_10['PUISSANCE Time'].min())
print(data_puissance_15_9_a_20_10['PUISSANCE Time'].max())

Courant : 
2022-09-15 14:49:12
2022-10-20 15:49:11
Débit Eau
2022-09-15 14:45:20
2022-10-20 15:45:17
Débit Solide
2022-09-15 14:40:58
2022-10-20 15:40:55
Puissance
2022-09-15 14:49:12
2022-10-20 17:01:55


In [76]:
def second_to_minute_2(data):
    columns = data.columns
    time_start = pd.to_datetime("2022-09-15 14:40:00")
    time_end = pd.to_datetime("2022-10-20 17:01:00")
    list_time = []
    list_value = []
    list_value_2 = []
    while(time_start <= time_end):
        interv = data[data[columns[0]].between(time_start,time_start + timedelta(minutes=1))]
        if interv.shape[0] == 0:
            list_value.append(float("nan"))
        else:
            list_value.append(interv[columns[1]].mean())
        list_time.append(time_start)
        if len(columns) == 4:
            list_value_2.append(interv[columns[3]].mean())
        time_start = time_start + timedelta(minutes=1)
    if len(columns) == 2:
        new_data = pd.DataFrame(data=zip(list_time,list_value),columns=[columns[0],columns[1]])
    elif len(columns) == 4:
        new_data = pd.DataFrame(data=zip(list_time,list_value,list_value_2),columns=[columns[0],columns[1],columns[3]])
    return new_data

In [77]:
new_data_courant_15_9_a_20_10 = second_to_minute_2(data_courant_15_9_a_20_10)
new_data_debit_eau_15_9_a_20_10 = second_to_minute_2(data_debit_eau_15_9_a_20_10)
new_data_debit_solide_15_9_a_20_10 = second_to_minute_2(data_debit_solide_15_9_a_20_10)
new_data_puissance_15_9_a_20_10 = second_to_minute_2(data_puissance_15_9_a_20_10)

In [80]:
print(new_data_courant_15_9_a_20_10)
print(new_data_debit_eau_15_9_a_20_10)
print(new_data_debit_solide_15_9_a_20_10)
print(new_data_puissance_15_9_a_20_10)

             COURANT Time  COURANT ValueY
0     2022-09-15 14:40:00             NaN
1     2022-09-15 14:41:00             NaN
2     2022-09-15 14:42:00             NaN
3     2022-09-15 14:43:00             NaN
4     2022-09-15 14:44:00             NaN
...                   ...             ...
50537 2022-10-20 16:57:00             NaN
50538 2022-10-20 16:58:00             NaN
50539 2022-10-20 16:59:00             NaN
50540 2022-10-20 17:00:00             NaN
50541 2022-10-20 17:01:00             NaN

[50542 rows x 2 columns]
           DEBIT EAU Time  DEBIT EAU ValueY
0     2022-09-15 14:40:00               NaN
1     2022-09-15 14:41:00               NaN
2     2022-09-15 14:42:00               NaN
3     2022-09-15 14:43:00               NaN
4     2022-09-15 14:44:00               NaN
...                   ...               ...
50537 2022-10-20 16:57:00               NaN
50538 2022-10-20 16:58:00               NaN
50539 2022-10-20 16:59:00               NaN
50540 2022-10-20 17:00:00     

In [83]:
new_data_courant_15_9_a_20_10["COURANT ValueY"].isna().sum()
new_data_courant_15_9_a_20_10.head(50)

,COURANT Time,COURANT ValueY
0,2022-09-15 14:40:00,NaN
1,2022-09-15 14:41:00,NaN
2,2022-09-15 14:42:00,NaN
3,2022-09-15 14:43:00,NaN
4,2022-09-15 14:44:00,NaN
5,2022-09-15 14:45:00,NaN
6,2022-09-15 14:46:00,NaN
7,2022-09-15 14:47:00,NaN
8,2022-09-15 14:48:00,NaN
9,2022-09-15 14:49:00,66.659241


In [84]:
new_data_courant_15_9_a_20_10.rename(columns={"COURANT Time": "Time"}, inplace=True)
new_data_debit_eau_15_9_a_20_10.rename(columns={"DEBIT EAU Time": "Time"}, inplace=True)
new_data_debit_solide_15_9_a_20_10.rename(columns={"SOLIDE Time": "Time"}, inplace=True)
new_data_puissance_15_9_a_20_10.rename(columns={"PUISSANCE Time": "Time"}, inplace=True)

In [85]:
data_15_09_a_20_10 = pd.merge(pd.merge(new_data_courant_15_9_a_20_10,new_data_debit_eau_15_9_a_20_10,on='Time'),pd.merge(new_data_debit_solide_15_9_a_20_10,new_data_puissance_15_9_a_20_10,on='Time'),on="Time")

In [86]:
data_15_09_a_20_10

,Time,COURANT ValueY,DEBIT EAU ValueY,SOLIDE ValueY,PUISSANCE ValueY
0,2022-09-15 14:40:00,NaN,NaN,114.791306,NaN
1,2022-09-15 14:41:00,NaN,NaN,116.841270,NaN
2,2022-09-15 14:42:00,NaN,NaN,114.797431,NaN
3,2022-09-15 14:43:00,NaN,NaN,114.255771,NaN
4,2022-09-15 14:44:00,NaN,NaN,117.610444,NaN
...,...,...,...,...,...
50537,2022-10-20 16:57:00,NaN,NaN,NaN,507.211636
50538,2022-10-20 16:58:00,NaN,NaN,NaN,511.160186
50539,2022-10-20 16:59:00,NaN,NaN,NaN,509.005283
50540,2022-10-20 17:00:00,NaN,NaN,NaN,510.360491


In [ ]:
data_15_09_a_20_10.to_csv("data_15_09_a_20_10.csv")

23_08 --> 15_09

Read csv files

In [3]:
data_courant_23_27_8 = pd.read_csv("De 23_08 à 15_09 2022\Courant\COURANT QA 23  27   8.csv", encoding='utf-16',sep="\t") 
data_courant_27_31_8 = pd.read_csv("De 23_08 à 15_09 2022\Courant\COURANT QA 27  31   8.csv", encoding='utf-16',sep="\t") 
data_courant_31_8_10_9 = pd.read_csv("De 23_08 à 15_09 2022\Courant\COURANT QA 31  8  10 9.csv", encoding='utf-16',sep="\t") 
data_courant_10_15_9 = pd.read_csv("De 23_08 à 15_09 2022\Courant\COURANT QA 10 15    9.csv", encoding='utf-16',sep="\t") 


data_Debit_eau_23_8_15_9 = pd.read_csv("De 23_08 à 15_09 2022\Débit Eau\DEBIT EAU ENTREE QA 23  8  15 9.csv",encoding='utf-16',sep="\t") 

data_Debit_solide_23_8_15_9 = pd.read_csv("De 23_08 à 15_09 2022\Débit Solide\DEBIT SOLIDE ENTREE QA 23  8 15 9.csv",encoding='utf-16', sep= '\t') 

data_refus_tamis_aout = pd.read_excel("De 23_08 à 15_09 2022\Refus Tamis\TB_D80_et_performances_Usine_Août_2022.xlsx")
data_refus_tamis_septembre_2 = pd.read_excel("De 23_08 à 15_09 2022\Refus Tamis\TB_D80_et_performances_Usine_Septembre_2022.xlsx")

data_puissance_QA_23_8_1_9 = pd.read_csv("De 23_08 à 15_09 2022\Puissance\PUISSACE    QA 23 8 1 9.csv",encoding='utf-16',sep="\t") 
data_puissance_QA_1_15_9 = pd.read_csv("De 23_08 à 15_09 2022\Puissance\PUISSANCE QA  1 15 9.csv",encoding='utf-16',sep="\t")

In [4]:
print("-----------------Courant-------------------------------")
print(data_courant_23_27_8)
print(data_courant_27_31_8)
print(data_courant_31_8_10_9)
print(data_courant_10_15_9)

print("-----------------Debit eau-------------------------------")
print(data_Debit_eau_23_8_15_9)


print("-----------------Debit solide-------------------------------")
print(data_Debit_solide_23_8_15_9)


print("-----------------Refus tamis-------------------------------")
print(data_refus_tamis_aout.head(10))
print(data_refus_tamis_septembre_2.head(10))

print("-----------------Puissance-------------------------------")
print(data_puissance_QA_23_8_1_9)
print(data_puissance_QA_1_15_9)

-----------------Courant-------------------------------
               COURANT Time    COURANT ValueY
0       23/08/2022 14:50:42  72,2747344970703
1       23/08/2022 14:50:42  71,5607604980469
2       23/08/2022 14:50:43  73,4544982910156
3       23/08/2022 14:50:43  66,8927230834961
4       23/08/2022 14:50:44  66,4031372070313
...                     ...               ...
619109  27/08/2022 04:50:39  63,5982360839844
619110  27/08/2022 04:50:40  64,2408142089844
619111  27/08/2022 04:50:40  64,2408142089844
619112  27/08/2022 04:50:41  71,6389617919922
619113  27/08/2022 04:50:41   63,258243560791

[619114 rows x 2 columns]
               COURANT Time    COURANT ValueY
0       27/08/2022 04:50:42  74,6886596679688
1       27/08/2022 04:50:42  74,6444549560547
2       27/08/2022 04:50:43  61,1979179382324
3       27/08/2022 04:50:43  68,0010833740234
4       27/08/2022 04:50:44  65,2097778320313
...                     ...               ...
691195  31/08/2022 04:50:39  66,60713195800

pre process refus tamis_aout & septembre_2

In [5]:
process_refu_tamis(data_refus_tamis_aout)
process_refu_tamis(data_refus_tamis_septembre_2)
print(data_refus_tamis_aout)
print(data_refus_tamis_septembre_2)

                  Jours Poste Refus Tamis +500 en % QA
3   2022-08-01 00:00:00     1                15.829722
4   2022-08-01 00:00:00     2                21.457873
5   2022-08-01 00:00:00     3                23.030496
6   2022-08-02 00:00:00     1                19.087793
7   2022-08-02 00:00:00     2                18.954439
..                  ...   ...                      ...
94  2022-08-31 00:00:00     2                19.498698
95  2022-08-31 00:00:00     3                 18.86958
96              MOYENNE   NaN                19.600263
97              MOYENNE   NaN                13.544802
98              MOYENNE   NaN                29.775759

[96 rows x 3 columns]
                  Jours Poste Refus Tamis +500 en % QA
3   2022-09-01 00:00:00     1                    29.03
4   2022-09-01 00:00:00     2                    22.29
5   2022-09-01 00:00:00     3                      NaN
6   2022-09-02 00:00:00     1                      NaN
7   2022-09-02 00:00:00     2             

print the types of every column

In [6]:
print(data_courant_23_27_8.dtypes)
print(data_courant_27_31_8.dtypes)
print(data_courant_31_8_10_9.dtypes)
print(data_courant_10_15_9.dtypes)

print(data_Debit_eau_23_8_15_9.dtypes)


print(data_Debit_solide_23_8_15_9.dtypes)


print(data_refus_tamis_aout.dtypes)
print(data_refus_tamis_septembre_2.dtypes)

print(data_puissance_QA_23_8_1_9.dtypes)
print(data_puissance_QA_1_15_9.dtypes)

COURANT Time      object
COURANT ValueY    object
dtype: object
COURANT Time      object
COURANT ValueY    object
dtype: object
COURANT Time      object
COURANT ValueY    object
dtype: object
COURANT Time      object
COURANT ValueY    object
dtype: object
DEBIT EAU Time      object
DEBIT EAU ValueY    object
dtype: object
SOLIDE Time      object
SOLIDE ValueY    object
SP_INT Time      object
SP_INT ValueY     int64
dtype: object
Jours                       object
Poste                       object
Refus Tamis +500 en % QA    object
dtype: object
Jours                       object
Poste                       object
Refus Tamis +500 en % QA    object
dtype: object
PUISSANCE Time      object
PUISSANCE ValueY    object
dtype: object
PUISSANCE Time      object
PUISSANCE ValueY    object
dtype: object


In [7]:
frames = (data_courant_23_27_8 , data_courant_27_31_8 , data_courant_31_8_10_9, data_courant_10_15_9)
data_courant_23_8_a_15_9 = pd.concat(frames)
data_courant_23_8_a_15_9

,COURANT Time,COURANT ValueY
0,23/08/2022 14:50:42,"72,2747344970703"
1,23/08/2022 14:50:42,"71,5607604980469"
2,23/08/2022 14:50:43,"73,4544982910156"
3,23/08/2022 14:50:43,"66,8927230834961"
4,23/08/2022 14:50:44,"66,4031372070313"
...,...,...
863995,15/09/2022 04:50:39,"59,402774810791"
863996,15/09/2022 04:50:40,"63,2038459777832"
863997,15/09/2022 04:50:40,"66,0291519165039"
863998,15/09/2022 04:50:41,"66,6887283325195"


In [8]:
frames_2 = (data_refus_tamis_aout , data_refus_tamis_septembre_2)
data_refus_tamis_23_8_a_15_9 = pd.concat(frames_2)
data_refus_tamis_23_8_a_15_9

,Jours,Poste,Refus Tamis +500 en % QA
3,2022-08-01 00:00:00,1,15.829722
4,2022-08-01 00:00:00,2,21.457873
5,2022-08-01 00:00:00,3,23.030496
6,2022-08-02 00:00:00,1,19.087793
7,2022-08-02 00:00:00,2,18.954439
...,...,...,...
91,2022-09-30 00:00:00,2,NaN
92,2022-09-30 00:00:00,3,NaN
93,MOYENNE,NaN,19.457157
94,MOYENNE,NaN,13.326848


In [9]:
frames_3 = (data_puissance_QA_23_8_1_9 , data_puissance_QA_1_15_9)
data_puissance_23_8_a_15_9 = pd.concat(frames_3)
data_puissance_23_8_a_15_9

,PUISSANCE Time,PUISSANCE ValueY
0,23/08/2022 13:46:49,"600,419738769531"
1,23/08/2022 13:46:49,"600,419738769531"
2,23/08/2022 13:46:50,"558,736083984375"
3,23/08/2022 13:46:50,"609,368713378906"
4,23/08/2022 13:46:51,"641,462158203125"
...,...,...
2419195,15/09/2022 04:50:39,"509,298645019531"
2419196,15/09/2022 04:50:40,"515,857299804688"
2419197,15/09/2022 04:50:40,"566,110961914063"
2419198,15/09/2022 04:50:41,"571,765869140625"


Convert the types from object to date_time

In [10]:
data_courant_23_8_a_15_9['COURANT Time'] = pd.to_datetime(data_courant_23_8_a_15_9['COURANT Time'], format="%d/%m/%Y %H:%M:%S")
data_Debit_eau_23_8_15_9['DEBIT EAU Time'] = pd.to_datetime(data_Debit_eau_23_8_15_9['DEBIT EAU Time'], format="%d/%m/%Y %H:%M:%S")
data_Debit_solide_23_8_15_9['SOLIDE Time'] = pd.to_datetime(data_Debit_solide_23_8_15_9['SOLIDE Time'], format="%d/%m/%Y %H:%M:%S")
data_puissance_23_8_a_15_9['PUISSANCE Time'] = pd.to_datetime(data_puissance_23_8_a_15_9['PUISSANCE Time'], format="%d/%m/%Y %H:%M:%S")
#data_refus_tamis_23_8_a_15_9['Jours'] = pd.to_datetime(data_refus_tamis_23_8_a_15_9['Jours'], format="%Y/%m/%d %H:%M:%S")

In [11]:
print(data_courant_23_8_a_15_9.dtypes)
print(data_Debit_eau_23_8_15_9.dtypes)
print(data_Debit_solide_23_8_15_9.dtypes)
print(data_puissance_23_8_a_15_9.dtypes)
#print(data_refus_tamis_23_8_a_15_9.dtypes)

COURANT Time      datetime64[ns]
COURANT ValueY            object
dtype: object
DEBIT EAU Time      datetime64[ns]
DEBIT EAU ValueY            object
dtype: object
SOLIDE Time      datetime64[ns]
SOLIDE ValueY            object
SP_INT Time              object
SP_INT ValueY             int64
dtype: object
PUISSANCE Time      datetime64[ns]
PUISSANCE ValueY            object
dtype: object


Replace the "," by "."

In [12]:
data_courant_23_8_a_15_9["COURANT ValueY"] = data_courant_23_8_a_15_9["COURANT ValueY"].apply(lambda x: x.replace("," , "."))
data_Debit_eau_23_8_15_9["DEBIT EAU ValueY"] = data_Debit_eau_23_8_15_9["DEBIT EAU ValueY"].apply(lambda x: x.replace("," , "."))
data_Debit_solide_23_8_15_9["SOLIDE ValueY"] = data_Debit_solide_23_8_15_9["SOLIDE ValueY"].apply(lambda x: x.replace("," , "."))
data_puissance_23_8_a_15_9["PUISSANCE ValueY"] = data_puissance_23_8_a_15_9["PUISSANCE ValueY"].apply(lambda x: x.replace("," , "."))

Change the type from "object" to "float" for the Value Columns

In [13]:
data_courant_23_8_a_15_9["COURANT ValueY"] = data_courant_23_8_a_15_9["COURANT ValueY"].astype(float)
data_Debit_eau_23_8_15_9["DEBIT EAU ValueY"] = data_Debit_eau_23_8_15_9["DEBIT EAU ValueY"].astype(float)
data_Debit_solide_23_8_15_9["SOLIDE ValueY"] = data_Debit_solide_23_8_15_9["SOLIDE ValueY"].astype(float)
data_puissance_23_8_a_15_9["PUISSANCE ValueY"] = data_puissance_23_8_a_15_9["PUISSANCE ValueY"].astype(float)

Shape of every csv file

In [14]:
print(data_courant_23_8_a_15_9.shape)
print(data_Debit_eau_23_8_15_9.shape)
print(data_Debit_solide_23_8_15_9.shape)
print(data_refus_tamis_23_8_a_15_9.shape)
print(data_puissance_23_8_a_15_9.shape)

(3902314, 2)
(995400, 2)
(995400, 4)
(189, 3)
(3909980, 2)


The max and the min

In [15]:
print("Courant : ")
print(data_courant_23_8_a_15_9['COURANT Time'].min())
print(data_courant_23_8_a_15_9['COURANT Time'].max())
print("Débit Eau")
print(data_Debit_eau_23_8_15_9['DEBIT EAU Time'].min())
print(data_Debit_eau_23_8_15_9['DEBIT EAU Time'].max())
print("Débit Solide")
print(data_Debit_solide_23_8_15_9['SOLIDE Time'].min())
print(data_Debit_solide_23_8_15_9['SOLIDE Time'].max())
print("Puissance")
print(data_puissance_23_8_a_15_9['PUISSANCE Time'].min())
print(data_puissance_23_8_a_15_9['PUISSANCE Time'].max())

Courant : 
2022-08-23 14:50:42
2022-09-15 04:50:41
Débit Eau
2022-08-23 17:02:30
2022-09-15 18:02:28
Débit Solide
2022-08-23 17:01:18
2022-09-15 18:01:16
Puissance
2022-08-23 13:46:49
2022-09-15 04:50:41


In [16]:
def second_to_minute_3(data):
    columns = data.columns
    time_start = pd.to_datetime("2022-08-23 13:46:00")
    time_end = pd.to_datetime("2022-09-15 18:02:00")
    list_time = []
    list_value = []
    list_value_2 = []
    while(time_start <= time_end):
        interv = data[data[columns[0]].between(time_start,time_start + timedelta(minutes=1))]
        if interv.shape[0] == 0:
            list_value.append(float("nan"))
        else:
            list_value.append(interv[columns[1]].mean())
        list_time.append(time_start)
        if len(columns) == 4:
            list_value_2.append(interv[columns[3]].mean())
        time_start = time_start + timedelta(minutes=1)
    if len(columns) == 2:
        new_data = pd.DataFrame(data=zip(list_time,list_value),columns=[columns[0],columns[1]])
    elif len(columns) == 4:
        new_data = pd.DataFrame(data=zip(list_time,list_value,list_value_2),columns=[columns[0],columns[1],columns[3]])
    return new_data

In [17]:
new_data_courant_23_8_a_15_9 = second_to_minute_3(data_courant_23_8_a_15_9)
new_data_Debit_eau_23_8_15_9 = second_to_minute_3(data_Debit_eau_23_8_15_9)
new_data_Debit_solide_23_8_15_9 = second_to_minute_3(data_Debit_solide_23_8_15_9)
new_data_puissance_23_8_a_15_9 = second_to_minute_3(data_puissance_23_8_a_15_9)

In [18]:
print(new_data_courant_23_8_a_15_9)
print(new_data_Debit_eau_23_8_15_9)
print(new_data_Debit_solide_23_8_15_9)
print(new_data_puissance_23_8_a_15_9)

             COURANT Time  COURANT ValueY
0     2022-08-23 13:46:00             NaN
1     2022-08-23 13:47:00             NaN
2     2022-08-23 13:48:00             NaN
3     2022-08-23 13:49:00             NaN
4     2022-08-23 13:50:00             NaN
...                   ...             ...
33372 2022-09-15 17:58:00             NaN
33373 2022-09-15 17:59:00             NaN
33374 2022-09-15 18:00:00             NaN
33375 2022-09-15 18:01:00             NaN
33376 2022-09-15 18:02:00             NaN

[33377 rows x 2 columns]
           DEBIT EAU Time  DEBIT EAU ValueY
0     2022-08-23 13:46:00               NaN
1     2022-08-23 13:47:00               NaN
2     2022-08-23 13:48:00               NaN
3     2022-08-23 13:49:00               NaN
4     2022-08-23 13:50:00               NaN
...                   ...               ...
33372 2022-09-15 17:58:00         11.085793
33373 2022-09-15 17:59:00         11.401396
33374 2022-09-15 18:00:00         11.526471
33375 2022-09-15 18:01:00     

In [ ]:
print(new_data_courant_23_8_a_15_9["COURANT ValueY"].isna().sum())
#new_data_courant_23_8_a_15_9.head(50)

In [20]:
new_data_courant_23_8_a_15_9.rename(columns={"COURANT Time": "Time"}, inplace=True)
new_data_Debit_eau_23_8_15_9.rename(columns={"DEBIT EAU Time": "Time"}, inplace=True)
new_data_Debit_solide_23_8_15_9.rename(columns={"SOLIDE Time": "Time"}, inplace=True)
new_data_puissance_23_8_a_15_9.rename(columns={"PUISSANCE Time": "Time"}, inplace=True)

In [21]:
data_23_08_a_15_09 = pd.merge(pd.merge(new_data_courant_23_8_a_15_9,new_data_Debit_eau_23_8_15_9,on='Time'),pd.merge(new_data_Debit_solide_23_8_15_9,new_data_puissance_23_8_a_15_9,on='Time'),on="Time")

In [22]:
data_23_08_a_15_09

,Time,COURANT ValueY,DEBIT EAU ValueY,SOLIDE ValueY,SP_INT ValueY,PUISSANCE ValueY
0,2022-08-23 13:46:00,NaN,NaN,NaN,NaN,586.605394
1,2022-08-23 13:47:00,NaN,NaN,NaN,NaN,592.525067
2,2022-08-23 13:48:00,NaN,NaN,NaN,NaN,583.371907
3,2022-08-23 13:49:00,NaN,NaN,NaN,NaN,592.231900
4,2022-08-23 13:50:00,NaN,NaN,NaN,NaN,580.151609
...,...,...,...,...,...,...
33372,2022-09-15 17:58:00,NaN,11.085793,116.198686,115.0,NaN
33373,2022-09-15 17:59:00,NaN,11.401396,114.182265,115.0,NaN
33374,2022-09-15 18:00:00,NaN,11.526471,112.822137,115.0,NaN
33375,2022-09-15 18:01:00,NaN,11.307124,116.986561,115.0,NaN


In [23]:
data_23_08_a_15_09.to_csv("data_23_08_a_15_09.csv")

28_01 --> 03_02

Read csv files

In [3]:
data_courant_28_1_a_03_2 = pd.read_csv("De 28_01 à 03_02 2022\Courant\COURANT BROYEUR QA.csv", encoding='utf-16',sep="\t") 

data_Debit_eau_28_1_a_03_2 = pd.read_csv("De 28_01 à 03_02 2022\Débit Eau\DEBIT EAU ENTREE QA.csv",encoding='utf-16',sep="\t") 

data_Debit_solide_28_1_a_03_2 = pd.read_csv("De 28_01 à 03_02 2022\Débit Solide\DEBIT SOLIDE QA.csv",encoding='utf-16', sep= '\t') 

data_refus_tamis_janvier = pd.read_excel("De 28_01 à 03_02 2022\Refus Tamis\TB_D80_et_performances_Usine_Janvier_2022.xlsx")
data_refus_tamis_fevrier = pd.read_excel("De 28_01 à 03_02 2022\Refus Tamis\TB_D80_et_performances_Usine_Février_2022.xlsx")

data_puissance_QA_28_1_03_2 = pd.read_csv("De 28_01 à 03_02 2022\Puissance\PUISSANCE BROYEUR QA.csv",encoding='utf-16',sep="\t") 

In [4]:
print("-----------------Courant-------------------------------")
print(data_courant_28_1_a_03_2)


print("-----------------Debit eau-------------------------------")
print(data_Debit_eau_28_1_a_03_2)


print("-----------------Debit solide-------------------------------")
print(data_Debit_solide_28_1_a_03_2)


print("-----------------Refus tamis-------------------------------")
print(data_refus_tamis_janvier.head(10))
print(data_refus_tamis_fevrier.head(10))

print("-----------------Puissance-------------------------------")
print(data_puissance_QA_28_1_03_2)

-----------------Courant-------------------------------
                COURANT Time    COURANT ValueY
0        28/01/2022 00:00:08  67,2531127929688
1        28/01/2022 00:00:08  67,2531127929688
2        28/01/2022 00:00:09  67,2531127929688
3        28/01/2022 00:00:09  67,2531127929688
4        28/01/2022 00:00:10  67,2531127929688
...                      ...               ...
1326585  04/02/2022 17:00:05  71,7919616699219
1326586  04/02/2022 17:00:06   62,867259979248
1326587  04/02/2022 17:00:06  61,2081146240234
1326588  04/02/2022 17:00:07  72,0775451660156
1326589  04/02/2022 17:00:07  66,6275329589844

[1326590 rows x 2 columns]
-----------------Debit eau-------------------------------
             DEBIT EAU Time  DEBIT EAU ValueY     SP_INT_AUTO Time  \
0       28/01/2022 00:00:57  10,1851854324341  28/01/2022 00:00:57   
1       28/01/2022 00:00:59  10,1851854324341  28/01/2022 00:00:59   
2       28/01/2022 00:01:01  10,1851854324341  28/01/2022 00:01:01   
3       28/01/

In [5]:
process_refu_tamis(data_refus_tamis_janvier)
process_refu_tamis(data_refus_tamis_fevrier)
print(data_refus_tamis_janvier)
print(data_refus_tamis_fevrier)

                  Jours Poste Refus Tamis +500 en % QA
3   2022-01-01 00:00:00     1                      NaN
4   2022-01-01 00:00:00     2                      NaN
5   2022-01-01 00:00:00     3                    25.81
6   2022-01-02 00:00:00     1                30.733229
7   2022-01-02 00:00:00     2                25.703125
..                  ...   ...                      ...
94  2022-01-31 00:00:00     2                29.193206
95  2022-01-31 00:00:00     3                31.105089
96              MOYENNE   NaN                25.956199
97              MOYENNE   NaN                15.255906
98              MOYENNE   NaN                 35.46169

[96 rows x 3 columns]
                  Jours Poste Refus Tamis +500 en % QA
3   2022-02-01 00:00:00     1                26.205511
4   2022-02-01 00:00:00     2                27.455296
5   2022-02-01 00:00:00     3                29.126214
6   2022-02-02 00:00:00     1                18.786127
7   2022-02-02 00:00:00     2             

print the types of every column

In [6]:
print(data_courant_28_1_a_03_2.dtypes)


print(data_Debit_eau_28_1_a_03_2.dtypes)


print(data_Debit_solide_28_1_a_03_2.dtypes)


print(data_refus_tamis_janvier.dtypes)
print(data_refus_tamis_fevrier.dtypes)

print(data_puissance_QA_28_1_03_2.dtypes)

COURANT Time      object
COURANT ValueY    object
dtype: object
DEBIT EAU Time        object
DEBIT EAU ValueY      object
SP_INT_AUTO Time      object
SP_INT_AUTO ValueY    object
dtype: object
SOLIDE Time      object
SOLIDE ValueY    object
SP_INT Time      object
SP_INT ValueY     int64
dtype: object
Jours                       object
Poste                       object
Refus Tamis +500 en % QA    object
dtype: object
Jours                       object
Poste                       object
Refus Tamis +500 en % QA    object
dtype: object
PUISSANCE Time      object
PUISSANCE ValueY    object
dtype: object


In [7]:
frames = (data_refus_tamis_janvier , data_refus_tamis_fevrier)
data_refus_tamis_28_1_03_2 = pd.concat(frames)
data_refus_tamis_28_1_03_2

,Jours,Poste,Refus Tamis +500 en % QA
3,2022-01-01 00:00:00,1,NaN
4,2022-01-01 00:00:00,2,NaN
5,2022-01-01 00:00:00,3,25.81
6,2022-01-02 00:00:00,1,30.733229
7,2022-01-02 00:00:00,2,25.703125
...,...,...,...
85,2022-02-28 00:00:00,2,33.198507
86,2022-02-28 00:00:00,3,28.062157
87,MOYENNE,NaN,24.345225
88,MOYENNE,NaN,15.627498


In [8]:
data_courant_28_1_a_03_2['COURANT Time'] = pd.to_datetime(data_courant_28_1_a_03_2['COURANT Time'], format="%d/%m/%Y %H:%M:%S")
data_Debit_eau_28_1_a_03_2['DEBIT EAU Time'] = pd.to_datetime(data_Debit_eau_28_1_a_03_2['DEBIT EAU Time'], format="%d/%m/%Y %H:%M:%S")
data_Debit_solide_28_1_a_03_2['SOLIDE Time'] = pd.to_datetime(data_Debit_solide_28_1_a_03_2['SOLIDE Time'], format="%d/%m/%Y %H:%M:%S")
data_puissance_QA_28_1_03_2['PUISSANCE Time'] = pd.to_datetime(data_puissance_QA_28_1_03_2['PUISSANCE Time'], format="%d/%m/%Y %H:%M:%S")
#data_refus_tamis_28_1_03_2['Jours'] = pd.to_datetime(data_refus_tamis_28_1_03_2['Jours'], format="%Y/%m/%d %H:%M:%S")

In [9]:
print(data_courant_28_1_a_03_2.dtypes)
print(data_Debit_eau_28_1_a_03_2.dtypes)
print(data_Debit_solide_28_1_a_03_2.dtypes)
print(data_puissance_QA_28_1_03_2.dtypes)
#print(data_refus_tamis_28_1_03_2.dtypes)

COURANT Time      datetime64[ns]
COURANT ValueY            object
dtype: object
DEBIT EAU Time        datetime64[ns]
DEBIT EAU ValueY              object
SP_INT_AUTO Time              object
SP_INT_AUTO ValueY            object
dtype: object
SOLIDE Time      datetime64[ns]
SOLIDE ValueY            object
SP_INT Time              object
SP_INT ValueY             int64
dtype: object
PUISSANCE Time      datetime64[ns]
PUISSANCE ValueY            object
dtype: object


Replace the "," by "."

In [10]:
data_courant_28_1_a_03_2["COURANT ValueY"] = data_courant_28_1_a_03_2["COURANT ValueY"].apply(lambda x: x.replace("," , "."))
data_Debit_eau_28_1_a_03_2["DEBIT EAU ValueY"] = data_Debit_eau_28_1_a_03_2["DEBIT EAU ValueY"].apply(lambda x: x.replace("," , "."))
data_Debit_eau_28_1_a_03_2["SP_INT_AUTO ValueY"] = data_Debit_eau_28_1_a_03_2["SP_INT_AUTO ValueY"].apply(lambda x: x.replace("," , "."))
data_Debit_solide_28_1_a_03_2["SOLIDE ValueY"] = data_Debit_solide_28_1_a_03_2["SOLIDE ValueY"].apply(lambda x: x.replace("," , "."))
data_puissance_QA_28_1_03_2["PUISSANCE ValueY"] = data_puissance_QA_28_1_03_2["PUISSANCE ValueY"].apply(lambda x: x.replace("," , "."))

Change the type from "object" to "float" for the Value Columns

In [11]:
data_courant_28_1_a_03_2["COURANT ValueY"] = data_courant_28_1_a_03_2["COURANT ValueY"].astype(float)
data_Debit_eau_28_1_a_03_2["DEBIT EAU ValueY"] = data_Debit_eau_28_1_a_03_2["DEBIT EAU ValueY"].astype(float)
data_Debit_eau_28_1_a_03_2["SP_INT_AUTO ValueY"] = data_Debit_eau_28_1_a_03_2["SP_INT_AUTO ValueY"].astype(float)
data_Debit_solide_28_1_a_03_2["SOLIDE ValueY"] = data_Debit_solide_28_1_a_03_2["SOLIDE ValueY"].astype(float)
data_puissance_QA_28_1_03_2["PUISSANCE ValueY"] = data_puissance_QA_28_1_03_2["PUISSANCE ValueY"].astype(float)

In [12]:
print(data_courant_28_1_a_03_2.dtypes)


print(data_Debit_eau_28_1_a_03_2.dtypes)


print(data_Debit_solide_28_1_a_03_2.dtypes)


print(data_refus_tamis_janvier.dtypes)
print(data_refus_tamis_fevrier.dtypes)

print(data_puissance_QA_28_1_03_2.dtypes)

COURANT Time      datetime64[ns]
COURANT ValueY           float64
dtype: object
DEBIT EAU Time        datetime64[ns]
DEBIT EAU ValueY             float64
SP_INT_AUTO Time              object
SP_INT_AUTO ValueY           float64
dtype: object
SOLIDE Time      datetime64[ns]
SOLIDE ValueY           float64
SP_INT Time              object
SP_INT ValueY             int64
dtype: object
Jours                       object
Poste                       object
Refus Tamis +500 en % QA    object
dtype: object
Jours                       object
Poste                       object
Refus Tamis +500 en % QA    object
dtype: object
PUISSANCE Time      datetime64[ns]
PUISSANCE ValueY           float64
dtype: object


data_puissance_QA_28_1_03_2

In [13]:
print(data_courant_28_1_a_03_2.shape)
print(data_Debit_eau_28_1_a_03_2.shape)
print(data_Debit_solide_28_1_a_03_2.shape)
print(data_refus_tamis_28_1_03_2.shape)
print(data_puissance_QA_28_1_03_2.shape)

(1326590, 2)
(332248, 4)
(304830, 4)
(183, 3)
(1326590, 2)


In [14]:
print("Courant : ")
print(data_courant_28_1_a_03_2['COURANT Time'].min())
print(data_courant_28_1_a_03_2['COURANT Time'].max())
print("Débit Eau")
print(data_Debit_eau_28_1_a_03_2['DEBIT EAU Time'].min())
print(data_Debit_eau_28_1_a_03_2['DEBIT EAU Time'].max())
print("Débit Solide")
print(data_Debit_solide_28_1_a_03_2['SOLIDE Time'].min())
print(data_Debit_solide_28_1_a_03_2['SOLIDE Time'].max())
print("Puissance")
print(data_puissance_QA_28_1_03_2['PUISSANCE Time'].min())
print(data_puissance_QA_28_1_03_2['PUISSANCE Time'].max())

Courant : 
2022-01-28 00:00:08
2022-02-04 17:00:07
Débit Eau
2022-01-28 00:00:57
2022-02-04 17:00:55
Débit Solide
2022-01-28 00:00:01
2022-02-04 01:20:59
Puissance
2022-01-28 00:00:08
2022-02-04 17:00:07


In [15]:
def second_to_minute_4(data):
    columns = data.columns
    time_start = pd.to_datetime("2022-01-28 00:00:00")
    time_end = pd.to_datetime("2022-02-04 17:00:00")
    list_time = []
    list_value = []
    list_value_2 = []
    while(time_start <= time_end):
        interv = data[data[columns[0]].between(time_start,time_start + timedelta(minutes=1))]
        if interv.shape[0] == 0:
            list_value.append(float("nan"))
        else:
            list_value.append(interv[columns[1]].mean())
        list_time.append(time_start)
        if len(columns) == 4:
            list_value_2.append(interv[columns[3]].mean())
        time_start = time_start + timedelta(minutes=1)
    if len(columns) == 2:
        new_data = pd.DataFrame(data=zip(list_time,list_value),columns=[columns[0],columns[1]])
    elif len(columns) == 4:
        new_data = pd.DataFrame(data=zip(list_time,list_value,list_value_2),columns=[columns[0],columns[1],columns[3]])
    return new_data

In [16]:
new_data_courant_28_1_a_03_2 = second_to_minute_4(data_courant_28_1_a_03_2)
new_data_Debit_eau_28_1_a_03_2 = second_to_minute_4(data_Debit_eau_28_1_a_03_2)
new_data_Debit_solide_28_1_a_03_2 = second_to_minute_4(data_Debit_solide_28_1_a_03_2)
new_data_puissance_28_1_a_03_2 = second_to_minute_4(data_puissance_QA_28_1_03_2)

In [17]:
print(new_data_courant_28_1_a_03_2)
print(new_data_Debit_eau_28_1_a_03_2)
print(new_data_Debit_solide_28_1_a_03_2)
print(new_data_puissance_28_1_a_03_2)

             COURANT Time  COURANT ValueY
0     2022-01-28 00:00:00       67.253113
1     2022-01-28 00:01:00       67.253113
2     2022-01-28 00:02:00       67.253113
3     2022-01-28 00:03:00       67.253113
4     2022-01-28 00:04:00       67.253113
...                   ...             ...
11096 2022-02-04 16:56:00       66.443994
11097 2022-02-04 16:57:00       66.702970
11098 2022-02-04 16:58:00       66.539776
11099 2022-02-04 16:59:00       67.152340
11100 2022-02-04 17:00:00       67.503427

[11101 rows x 2 columns]
           DEBIT EAU Time  DEBIT EAU ValueY  SP_INT_AUTO ValueY
0     2022-01-28 00:00:00         10.185185                10.0
1     2022-01-28 00:01:00         10.185185                10.0
2     2022-01-28 00:02:00         10.185185                10.0
3     2022-01-28 00:03:00         10.185185                10.0
4     2022-01-28 00:04:00         10.185185                10.0
...                   ...               ...                 ...
11096 2022-02-04 16:56

In [18]:
new_data_courant_28_1_a_03_2.rename(columns={"COURANT Time": "Time"}, inplace=True)
new_data_Debit_eau_28_1_a_03_2.rename(columns={"DEBIT EAU Time": "Time"}, inplace=True)
new_data_Debit_solide_28_1_a_03_2.rename(columns={"SOLIDE Time": "Time"}, inplace=True)
new_data_puissance_28_1_a_03_2.rename(columns={"PUISSANCE Time": "Time"}, inplace=True)

In [19]:
data_28_01_a_03_02 = pd.merge(pd.merge(new_data_courant_28_1_a_03_2,new_data_Debit_eau_28_1_a_03_2,on='Time'),pd.merge(new_data_Debit_solide_28_1_a_03_2,new_data_puissance_28_1_a_03_2,on='Time'),on="Time")

In [20]:
data_28_01_a_03_02

,Time,COURANT ValueY,DEBIT EAU ValueY,SP_INT_AUTO ValueY,SOLIDE ValueY,SP_INT ValueY,PUISSANCE ValueY
0,2022-01-28 00:00:00,67.253113,10.185185,10.0,121.003326,120.0,576.604736
1,2022-01-28 00:01:00,67.253113,10.185185,10.0,121.003326,120.0,576.604736
2,2022-01-28 00:02:00,67.253113,10.185185,10.0,121.003326,120.0,576.604736
3,2022-01-28 00:03:00,67.253113,10.185185,10.0,121.003326,120.0,576.604736
4,2022-01-28 00:04:00,67.253113,10.185185,10.0,121.003326,120.0,576.604736
...,...,...,...,...,...,...,...
11096,2022-02-04 16:56:00,66.443994,9.648076,9.5,NaN,NaN,562.365687
11097,2022-02-04 16:57:00,66.702970,9.724392,9.5,NaN,NaN,570.214521
11098,2022-02-04 16:58:00,66.539776,9.275415,9.5,NaN,NaN,565.358756
11099,2022-02-04 16:59:00,67.152340,8.460648,9.5,NaN,NaN,575.820535


In [21]:
data_28_01_a_03_02.to_csv("data_28_01_a_03_02.csv")